In [1]:
from utils import get_tables
import json
import os
import pandas as pd

In [2]:
PDF = "34. GLEN-2023-Sustainability-Report.pdf"
INCLUDE_TABLES = True
EXPORT_TO_XLSX = True

In [9]:
PDF = os.path.basename(PDF)
REPORT_JSON_PATH = f"./data/ReportJson/{os.path.splitext(PDF)[0]}.json"
with open(REPORT_JSON_PATH, 'r') as f: 
    document = json.load(f)

OUT_JSON_PATH = f"./out/json/{os.path.splitext(PDF)[0]}_sources.json"
out_data = {}
if os.path.exists(OUT_JSON_PATH): 
    with open(OUT_JSON_PATH, 'r') as f:
        out_data = json.load(f)

table_data = get_tables.main(document)
out_data['tables'] = table_data

table_refs = set(table_data['all table refs'])
relevant_tables = [t for t in document['tables'] if t['self_ref'] in table_refs]

if INCLUDE_TABLES: 
    out_data['tables']['tables'] = relevant_tables

with open(OUT_JSON_PATH, 'w') as f: 
    json.dump(out_data, f)

if EXPORT_TO_XLSX:
    with pd.ExcelWriter(f"./out/xlsx/{os.path.splitext(PDF)[0]}_tables.xlsx") as writer: 
        for i, t in enumerate(relevant_tables):
            df = get_tables.extract_table_to_df(t) 
            df.to_excel(writer, sheet_name=f"Sheet_{i}", index=False)